In [5]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

In [6]:
FIGSIZE = (10, 6)
DPI = 300
TRANSPARENT = True

In [13]:
MODEL_DIRS = {
    "conspiracy": Path(".../conspiracy/4_model/model_4/model_data"),
    "crypto": Path(".../crypto/4_model/model_4/model_data"),
    "politics": Path(".../politics/4_model/model_4/model_data"),
}


OUTDIR = Path("C:/Users/snuzz/OneDrive/Documents/PhD/abstracts, applications, papers, reports/thread_size_prediction/poster/figs")
OUTDIR.mkdir(parents=True, exist_ok=True)

In [12]:
# Constants
SUBREDDIT_LABELS = {
    "conspiracy": "r/Conspiracy",
    "crypto": "r/CryptoCurrency",
    "politics": "r/politics",
}

feat_names = {
    "author_frequency": "Author activity",
    "domain_frequency": "Domain activity",
    "question_ratio": "Question ratio",
    "subject_length": "Post length",
    "domain_pagerank": "Domain centrality",
    "hour": "Hour",
}

COLORS = {
    "dark_purple": "#361a54",
    "off_white": "#fafafa",
    "bright_purple": "#993bff",
    "heritage_blue": "#30d6ff",
}

TEXT_COLOR_DARK = COLORS["dark_purple"]
TEXT_COLOR_LIGHT = COLORS["off_white"]

UCL_CMAP = LinearSegmentedColormap.from_list(
    "ucl_shap",
    [COLORS["heritage_blue"], COLORS["bright_purple"]]
)

LETTER_LOOKUP = {
    0: "(a)",
    1: "(b)",
    2: "(c)",
    3: "(d)",
}

In [9]:
TEXT_THEME = "dark"   # "dark" -> dark purple text/axes
                      # "light" -> off white text/axes


MAX_DISPLAY = 10
FIGSIZE = (8, 6)
DPI = 300
TRANSPARENT = True
SAMPLE_N = None   # or e.g. 1000 if you want to subsample for speed

In [10]:
def style_shap_axes(main_ax, colorbar_ax, text_color):
    """Apply poster styling to SHAP summary plot axes."""
    main_ax.set_xlabel("SHAP value", fontsize=11, color=text_color)
    main_ax.set_ylabel("Feature", fontsize=11, color=text_color)
    main_ax.tick_params(axis="both", colors=text_color, labelsize=10)

    for spine in main_ax.spines.values():
        spine.set_edgecolor(text_color)

    if colorbar_ax is not None:
        colorbar_ax.set_ylabel("Feature value", fontsize=11, color=text_color)
        colorbar_ax.tick_params(axis="y", colors=text_color, labelsize=10)
        for spine in colorbar_ax.spines.values():
            spine.set_edgecolor(text_color)


def make_stage1_shap_plot(
    sub_shap_dict,
    outfile,
    title,
    text_color,
    cmap,
    max_display=MAX_DISPLAY,
    figsize=FIGSIZE,
    dpi=DPI,
    transparent=TRANSPARENT,
):
    """
    Make one Stage 1 SHAP beeswarm/summary plot from saved shap_plot_data.jl.

    Expected keys in sub_shap_dict:
        - 'shap_val': array-like of shape (n_samples, n_features)
        - 'feat_name': pandas DataFrame with feature columns
    """
    X = sub_shap_dict["feat_name"].copy()
    shap_vals = sub_shap_dict["shap_val"]

    # Optional renaming for poster-friendly labels
    rename_map = {c: feat_names.get(c, c) for c in X.columns}
    X = X.rename(columns=rename_map)

    plt.close("all")
    plt.figure(figsize=figsize)

    shap.summary_plot(
        shap_vals,
        X,
        plot_type="dot",
        show=False,
        cmap=cmap,
        max_display=max_display,
        feature_names=list(X.columns),
    )

    fig = plt.gcf()
    fig.patch.set_alpha(0)

    axes = fig.get_axes()

    # Main SHAP axis: usually the one with the SHAP x-label
    main_ax = None
    colorbar_ax = None

    for ax in axes:
        xlabel = ax.get_xlabel()
        ylabel = ax.get_ylabel()

        if xlabel == "SHAP value":
            main_ax = ax
        elif "Feature value" in ylabel:
            colorbar_ax = ax

    if main_ax is None:
        # fallback in case SHAP version labels slightly differently
        for ax in axes:
            if isinstance(ax, plt.Axes):
                main_ax = ax
                break

    if main_ax is not None:
        main_ax.set_title(title, loc="left", fontsize=12, weight="bold", color=text_color)
        main_ax.patch.set_alpha(0)

    if colorbar_ax is not None:
        colorbar_ax.patch.set_alpha(0)

    style_shap_axes(main_ax, colorbar_ax, text_color)

    plt.tight_layout()
    plt.savefig(
        outfile,
        dpi=dpi,
        format="png",
        bbox_inches="tight",
        transparent=transparent,
    )
    plt.close(fig)


def make_all_stage1_poster_shap_plots(model_dirs, outdir):
    """Generate both dark-text and light-text versions for each subreddit."""
    for i, (sub, mod_dir) in enumerate(model_dirs.items()):
        shap_path = mod_dir / "shap_plot_data.jl"
        shap_data = joblib.load(shap_path)

        title = f"{LETTER_LOOKUP[i]} {SUBREDDIT_LABELS[sub]}"

        dark_out = outdir / f"{sub}_thread_start_shap_dark.png"
        light_out = outdir / f"{sub}_thread_start_shap_light.png"

        make_stage1_shap_plot(
            sub_shap_dict=shap_data,
            outfile=dark_out,
            title=title,
            text_color=TEXT_COLOR_DARK,
            cmap=UCL_CMAP,
        )

        make_stage1_shap_plot(
            sub_shap_dict=shap_data,
            outfile=light_out,
            title=title,
            text_color=TEXT_COLOR_LIGHT,
            cmap=UCL_CMAP,
        )

        print(f"Saved: {dark_out}")
        print(f"Saved: {light_out}")

In [ ]:
make_all_stage1_poster_shap_plots(MODEL_DIRS, OUTDIR)